In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/Users/Jiyeon/Desktop/ftp/fertility-treatment-prediction/data/raw'

train = pd.read_csv('train_fe_v3.csv')
test  = pd.read_csv('test_fe_v3.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

print('Train:', train.shape, '/ Test:', test.shape)

Train: (256351, 192) / Test: (90067, 191)


In [2]:
# feature_refinement + 신규 피처 적용 (시술시기 교호작용 제외)
# 중요도 0 제거
ZERO_IMP_COLS = [
    '배아생성_제로_플래그','배아생성_실패_플래그','난자수집_실패_플래그',
    '난자저장용_포함','배반포이식_여부','불임 원인 - 여성 요인',
    'IVF_배반포_조합','불임 원인 - 정자 면역학적 요인','high_resp_freezeall',
]
drop_cols = [c for c in ZERO_IMP_COLS if c in train.columns]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=drop_cols, inplace=True)

# 기존 교호작용
for df in [train, test]:
    emb_n   = df['이식된 배아 수'].fillna(0)
    emb_log = df['이식된 배아 수_log'].fillna(0)
    age     = df['나이_순서'].fillna(0)
    quality = df['배아품질_종합점수'].fillna(0)
    is_ivf  = (df['시술 유형'] == 'IVF').astype(int)
    is_tx   = df['이식배아_있음'].fillna(0)
    tx_day  = df['배아 이식 경과일'].replace(-1, np.nan).fillna(0)
    culture = df['배양_기간_일'].replace(-1, np.nan).fillna(0)
    stored  = df['저장된 배아 수'].fillna(0)

    df['이식배아수_나이_교호']     = emb_n * age
    df['이식배아log_나이_교호']    = emb_log * age
    df['이식배아수_품질_교호']     = emb_n * quality
    df['이식배아log_품질_교호']    = emb_log * quality
    df['이식배아수_IVF_교호']      = emb_n * is_ivf
    df['이식성공_품질_교호']       = is_tx * quality
    df['고령자가난자_품질_교호']   = df['자가난자_고령_조합'].fillna(0) * quality
    df['고위험3중_이식배아_교호']  = df['고위험_3중_조합'].fillna(0) * emb_n
    df['이식경과일_이식배아_교호'] = tx_day * emb_n
    df['이식배아수_출산경험_교호'] = emb_n * df['출산경험_있음'].fillna(0)
    df['나이_저장비율_교호']       = age * df['저장_비율'].fillna(0)
    df['잔여배아_품질_교호']       = df['잔여배아_수'].fillna(0) * quality
    df['이식배아수_제곱']          = emb_n ** 2
    df['이식배아_총배아_곱']       = emb_n * df['총 생성 배아 수'].fillna(0)
    df['단일배아이식_여부']        = (emb_n == 1).astype(int)
    df['2개배아이식_여부']         = (emb_n == 2).astype(int)

    # 시술 시기 코드 순서형 (단독 변수 — 교호작용 없이)
    time_code_order = (
        train['시술 시기 코드']
        .value_counts()
        .rank(ascending=False, method='first')
        .astype(int)
        .to_dict()
    )
    df['시술시기_순서형'] = df['시술 시기 코드'].map(time_code_order).fillna(-1).astype(int)

    # 신규 피처 (시술 시기 교호작용 2개 제외)
    df['이식여부_이식경과일_교호']  = is_tx * tx_day
    df['이식여부_배양기간_교호']    = is_tx * culture
    df['이식배아수_이식경과일_교호'] = emb_n * tx_day
    df['이식배아수_배양기간_교호']  = emb_n * culture
    df['이식배아수_저장배아_교호']  = emb_n * stored
    # ❌ 시술시기_나이_교호    제거 (랜덤 코드 기반 → 과적합)
    # ❌ 시술시기_이식배아_교호 제거 (랜덤 코드 기반 → 과적합)

print('피처 정제 완료')
print('제거된 피처: 시술시기_나이_교호, 시술시기_이식배아_교호')

피처 정제 완료
제거된 피처: 시술시기_나이_교호, 시술시기_이식배아_교호


In [3]:
# 피처 준비
feat_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

X      = train[feat_cols].copy()
y      = train[TARGET].copy()
X_test = test[feat_cols].copy()

cat_features = [
    c for c in feat_cols
    if X[c].dtype == 'object' or pd.api.types.is_string_dtype(X[c])
]
for col in cat_features:
    X[col]      = X[col].astype(str).fillna('missing')
    X_test[col] = X_test[col].astype(str).fillna('missing')

cat_feature_indices = [X.columns.get_loc(c) for c in cat_features]

X_lgb      = X.copy()
X_test_lgb = X_test.copy()
for col in cat_features:
    X_lgb[col]      = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_xgb      = X.copy()
X_test_xgb = X_test.copy()
X_xgb[cat_features]      = enc.fit_transform(X[cat_features])
X_test_xgb[cat_features] = enc.transform(X_test[cat_features])
num_features = [c for c in feat_cols if c not in cat_features]
medians = X_xgb[num_features].median()
X_xgb[num_features]      = X_xgb[num_features].fillna(medians)
X_test_xgb[num_features] = X_test_xgb[num_features].fillna(medians)

neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos

N_SPLITS = 5
SEED     = 42
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

print(f'피처: {len(feat_cols)}개 / 범주형: {len(cat_features)}개')
print(f'scale_pos_weight: {spw:.2f}')

피처: 203개 / 범주형: 21개
scale_pos_weight: 2.87


In [4]:
# 3모델 5-fold 학습 + 앙상블
cat_params = {
    'iterations': 794, 'learning_rate': 0.030094391673729362,
    'depth': 7, 'l2_leaf_reg': 7.433949857398995,
    'bagging_temperature': 0.3980358270898108,
    'random_strength': 1.3075975210328405, 'border_count': 108,
    'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'scale_pos_weight': spw, 'random_seed': SEED,
    'early_stopping_rounds': 50, 'verbose': False, 'use_best_model': True,
}
lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
    'num_leaves': 22, 'max_depth': 7, 'learning_rate': 0.02030926523583015,
    'n_estimators': 956, 'subsample': 0.9916893928795039,
    'colsample_bytree': 0.8760988294276582,
    'reg_alpha': 0.5158539052029842, 'reg_lambda': 0.05557192411355649,
    'min_child_samples': 98,
    'scale_pos_weight': spw, 'random_state': SEED, 'verbose': -1,
}
xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'hist',
    'max_depth': 4, 'learning_rate': 0.028817004923326044,
    'n_estimators': 835, 'subsample': 0.8755018207309047,
    'colsample_bytree': 0.8959610275285067,
    'reg_alpha': 0.9163081802804784, 'reg_lambda': 1.1472566862699578,
    'min_child_weight': 8, 'gamma': 0.728514692891076,
    'scale_pos_weight': spw, 'random_state': SEED,
    'verbosity': 0, 'early_stopping_rounds': 50,
}

oof_cat  = np.zeros(len(X))
oof_lgb  = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
test_cat = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr  = X.iloc[tr_idx].reset_index(drop=True)
    X_val = X.iloc[val_idx].reset_index(drop=True)
    y_tr  = y.iloc[tr_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)

    # CatBoost
    cb = CatBoostClassifier(**cat_params)
    cb.fit(Pool(X_tr, y_tr, cat_features=cat_feature_indices),
           eval_set=Pool(X_val, y_val, cat_features=cat_feature_indices))
    oof_cat[val_idx]  = cb.predict_proba(Pool(X_val, cat_features=cat_feature_indices))[:, 1]
    test_cat         += cb.predict_proba(Pool(X_test, cat_features=cat_feature_indices))[:, 1] / N_SPLITS

    # LightGBM
    lb = lgb.LGBMClassifier(**lgb_params)
    lb.fit(X_lgb.iloc[tr_idx], y_tr,
           eval_set=[(X_lgb.iloc[val_idx], y_val)],
           callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    oof_lgb[val_idx]  = lb.predict_proba(X_lgb.iloc[val_idx])[:, 1]
    test_lgb         += lb.predict_proba(X_test_lgb)[:, 1] / N_SPLITS

    # XGBoost
    xb = xgb.XGBClassifier(**xgb_params)
    xb.fit(X_xgb.iloc[tr_idx], y_tr,
           eval_set=[(X_xgb.iloc[val_idx], y_val)],
           verbose=False)
    oof_xgb[val_idx]  = xb.predict_proba(X_xgb.iloc[val_idx])[:, 1]
    test_xgb         += xb.predict_proba(X_test_xgb)[:, 1] / N_SPLITS

    cat_auc = roc_auc_score(y_val, oof_cat[val_idx])
    lgb_auc = roc_auc_score(y_val, oof_lgb[val_idx])
    xgb_auc = roc_auc_score(y_val, oof_xgb[val_idx])
    print(f'Fold {fold+1} | CAT: {cat_auc:.5f} | LGB: {lgb_auc:.5f} | XGB: {xgb_auc:.5f}')

print(f'\n✅ CatBoost OOF AUC : {roc_auc_score(y, oof_cat):.5f}  (이전: 0.74051)')
print(f'✅ LightGBM OOF AUC : {roc_auc_score(y, oof_lgb):.5f}  (이전: 0.73974)')
print(f'✅ XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.5f}  (이전: 0.74018)')

Fold 1 | CAT: 0.73844 | LGB: 0.73805 | XGB: 0.73857
Fold 2 | CAT: 0.74318 | LGB: 0.74200 | XGB: 0.74301
Fold 3 | CAT: 0.74076 | LGB: 0.74047 | XGB: 0.74033
Fold 4 | CAT: 0.73859 | LGB: 0.73813 | XGB: 0.73838
Fold 5 | CAT: 0.74083 | LGB: 0.74043 | XGB: 0.74078

✅ CatBoost OOF AUC : 0.74036  (이전: 0.74051)
✅ LightGBM OOF AUC : 0.73980  (이전: 0.73974)
✅ XGBoost  OOF AUC : 0.74020  (이전: 0.74018)


In [5]:
# 0.05 단위 가중치 탐색
best_auc_vote, best_w = 0.0, (0.6, 0.15, 0.25)
results = []

for w1 in np.arange(0.0, 1.05, 0.05):
    for w2 in np.arange(0.0, 1.05 - w1, 0.05):
        w3 = round(1 - w1 - w2, 2)
        if w3 < 0:
            continue
        blended = oof_cat * w1 + oof_lgb * w2 + oof_xgb * w3
        auc = roc_auc_score(y, blended)
        results.append((round(w1,2), round(w2,2), round(w3,2), auc))
        if auc > best_auc_vote:
            best_auc_vote = auc
            best_w = (w1, w2, w3)

results_df = pd.DataFrame(results, columns=['w_cat','w_lgb','w_xgb','oof_auc'])
print('=== 소프트 보팅 상위 10개 ===')
display(results_df.sort_values('oof_auc', ascending=False).head(10))

print(f'\n✅ 앙상블 최적 AUC  : {best_auc_vote:.5f}')
print(f'   최적 가중치        : CAT={best_w[0]:.2f}, LGB={best_w[1]:.2f}, XGB={best_w[2]:.2f}')
print(f'   이전 최고 (OOF)    : 0.74061')
print(f'   개선폭             : {best_auc_vote - 0.74061:+.5f}')
print(f'   리더보드 최고      : 0.74193')

=== 소프트 보팅 상위 10개 ===


,w_cat,w_lgb,w_xgb,oof_auc
178,0.55,0.10,0.35,0.740523
177,0.55,0.05,0.40,0.740522
188,0.60,0.10,0.30,0.740521
187,0.60,0.05,0.35,0.740521
167,0.50,0.10,0.40,0.740520
179,0.55,0.15,0.30,0.740520
186,0.60,0.00,0.40,0.740518
168,0.50,0.15,0.35,0.740518
166,0.50,0.05,0.45,0.740518
176,0.55,0.00,0.45,0.740517



✅ 앙상블 최적 AUC  : 0.74052
   최적 가중치        : CAT=0.55, LGB=0.10, XGB=0.35
   이전 최고 (OOF)    : 0.74061
   개선폭             : -0.00009
   리더보드 최고      : 0.74193


In [6]:
# 제출 파일 생성
test_blended = test_cat * best_w[0] + test_lgb * best_w[1] + test_xgb * best_w[2]

sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
submission = sample_sub.copy()
submission[sample_sub.columns[1]] = test_blended
submission.to_csv('submission_timecode_removed.csv', index=False, encoding='utf-8-sig')

print('저장 완료: submission_timecode_removed.csv')
print(f'\n최종 요약')
print(f'  CatBoost  : {roc_auc_score(y, oof_cat):.5f}')
print(f'  LightGBM  : {roc_auc_score(y, oof_lgb):.5f}')
print(f'  XGBoost   : {roc_auc_score(y, oof_xgb):.5f}')
print(f'  앙상블     : {best_auc_vote:.5f}')
print(f'  이전 최고 (OOF)      : 0.74061')
print(f'  이전 최고 (리더보드) : 0.74193')
display(submission.head())

저장 완료: submission_timecode_removed.csv

최종 요약
  CatBoost  : 0.74036
  LightGBM  : 0.73980
  XGBoost   : 0.74020
  앙상블     : 0.74052
  이전 최고 (OOF)      : 0.74061
  이전 최고 (리더보드) : 0.74193


,ID,probability
0,TEST_00000,0.002726
1,TEST_00001,0.013946
2,TEST_00002,0.342128
3,TEST_00003,0.248358
4,TEST_00004,0.726611
